# 30 · Quem assina pelo site sai mais?

Compara o canal de cobrança (site: `web_cartao` e `web_pix`; lojas: `app_store` e `google_play`)
nos três fatores: assinantes ativos, engajamento (níveis de `src/engajamento.py`, no corte) e churn
(taxa mensal = perdas ÷ ativas no início do mês, jul/2024 fora), separando plano e tipo de saída.

In [1]:
import sys
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import NIVEIS

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")

assin = pd.read_parquet(P / "assinaturas_limpa.parquet")
painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["mes", "user_id", "plano", "canal_cobranca", "ativa_inicio_mes", "perdida_no_mes", "motivo_fim"])
eng = pd.read_parquet(P / "engajamento_corte.parquet")
SITE = {"web_cartao": "site", "web_pix": "site", "app_store": "loja", "google_play": "loja"}
assin["onde"] = assin.canal_cobranca.map(SITE)
painel["onde"] = painel.canal_cobranca.map(SITE)
print(len(assin), "assinaturas |", len(painel), "linhas do painel |", len(eng), "ativos com nível")
print("assinaturas cujo canal do painel difere do retrato:", int((painel.groupby("user_id").canal_cobranca.nunique() > 1).sum()))

32583 assinaturas | 327771 linhas do painel | 14753 ativos com nível
assinaturas cujo canal do painel difere do retrato: 0


## 1. Ativos e saídas acumuladas

In [2]:
def resumo(col):
    t = assin.groupby(col).agg(assinaturas=("user_id", "size"), ativas=("ativa_no_corte", "sum"), pct_anual=("plano_atual", lambda s: (s == "anual").mean() * 100))
    t["sairam"] = t.assinaturas - t.ativas
    t["% das assinaturas"] = t.assinaturas / t.assinaturas.sum() * 100
    t["% das ativas"] = t.ativas / t.ativas.sum() * 100
    t["% saiu"] = t.sairam / t.assinaturas * 100
    return t
print(resumo("onde")); print(); print(resumo("canal_cobranca"))
print()
print("% que saiu, por canal e plano")
assin["saiu"] = ~assin.ativa_no_corte
print(assin.pivot_table(index="canal_cobranca", columns="plano_atual", values="saiu", aggfunc="mean") * 100)
print(assin.pivot_table(index="onde", columns="plano_atual", values="saiu", aggfunc="mean") * 100)

      assinaturas  ativas  pct_anual  sairam  % das assinaturas  % das ativas  % saiu
onde                                                                                 
loja        22118   10359       41.1   11759               67.9          70.2    53.2
site        10465    4394       37.7    6071               32.1          29.8    58.0

                assinaturas  ativas  pct_anual  sairam  % das assinaturas  % das ativas  % saiu
canal_cobranca                                                                                 
app_store             11170    5395       40.8    5775               34.3          36.6    51.7
google_play           10948    4964       41.4    5984               33.6          33.6    54.7
web_cartao             7087    2994       40.3    4093               21.8          20.3    57.8
web_pix                3378    1400       32.2    1978               10.4           9.5    58.6

% que saiu, por canal e plano
plano_atual     anual  mensal
canal_cobranca    

## 2. Taxa de churn mensal

In [3]:
p = painel[painel.mes >= "2024-08-01"].copy()
print("painel sem jul/2024:", len(p))
p["ano"] = p.mes.dt.year
p["involuntaria"] = p.perdida_no_mes & p.motivo_fim.isin(["falha_pagamento", "chargeback"])
p["voluntaria"] = p.perdida_no_mes & p.motivo_fim.isin(["renovacao_desligada", "reembolso"])
print("perdas:", int(p.perdida_no_mes.sum()), "| voluntárias:", int(p.voluntaria.sum()), "| involuntárias:", int(p.involuntaria.sum()))

def churn(d, por):
    g = d.groupby(por).agg(ativas_inicio=("ativa_inicio_mes", "sum"), perdas=("perdida_no_mes", "sum"), vol=("voluntaria", "sum"), invol=("involuntaria", "sum"))
    g["churn %"] = g.perdas / g.ativas_inicio * 100
    g["voluntário %"] = g.vol / g.ativas_inicio * 100
    g["involuntário %"] = g.invol / g.ativas_inicio * 100
    return g

d26 = p[p.ano == 2026]
print(); print("== 2026 (jan a set), site × loja"); print(churn(d26, "onde"))
print(); print("== 2026, por canal"); c26 = churn(d26, "canal_cobranca"); print(c26)
print(); print("== 2026, por plano e canal"); cp = churn(d26, ["plano", "canal_cobranca"]); print(cp)
print(); print("== 2026, por plano, site × loja"); print(churn(d26, ["plano", "onde"]))
print(); print("== churn % por canal e ano"); print(churn(p, ["canal_cobranca", "ano"])["churn %"].unstack())
print(); print("== mensal: churn % por canal e ano"); print(churn(p[p.plano == "mensal"], ["canal_cobranca", "ano"])["churn %"].unstack())
cp.to_csv(S / "canal_cobranca_churn_2026.csv", encoding="utf-8-sig")

painel sem jul/2024: 321927
perdas: 17543 | voluntárias: 16106 | involuntárias: 1437

== 2026 (jan a set), site × loja
      ativas_inicio  perdas   vol  invol  churn %  voluntário %  involuntário %
onde                                                                           
loja          87659    5546  5307    239      6.3           6.1             0.3
site          38188    2774  2380    394      7.3           6.2             1.0

== 2026, por canal


                ativas_inicio  perdas   vol  invol  churn %  voluntário %  involuntário %
canal_cobranca                                                                           
app_store               44822    2671  2545    126      6.0           5.7             0.3
google_play             42837    2875  2762    113      6.7           6.4             0.3
web_cartao              25787    1865  1534    331      7.2           5.9             1.3
web_pix                 12401     909   846     63      7.3           6.8             0.5

== 2026, por plano e canal
                       ativas_inicio  perdas   vol  invol  churn %  voluntário %  involuntário %
plano  canal_cobranca                                                                           
anual  app_store               25239     787   773     14      3.1           3.1             0.1
       google_play             24754     835   819     16      3.4           3.3             0.1
       web_cartao              15159     547

             ativas_inicio  perdas   vol  invol  churn %  voluntário %  involuntário %
plano  onde                                                                           
anual  loja          49993    1622  1592     30      3.2           3.2             0.1
       site          20943     732   675     57      3.5           3.2             0.3
mensal loja          37666    3924  3715    209     10.4           9.9             0.6
       site          17245    2042  1705    337     11.8           9.9             2.0

== churn % por canal e ano
ano             2024  2025  2026
canal_cobranca                  
app_store        4.6   5.4   6.0
google_play      4.6   5.4   6.7
web_cartao       5.0   6.1   7.2
web_pix          5.6   6.2   7.3

== mensal: churn % por canal e ano


ano             2024  2025  2026
canal_cobranca                  
app_store        8.1   8.6   9.6
google_play      8.1   8.4  11.3
web_cartao       9.2  10.2  12.4
web_pix          8.9   9.1  10.9


In [4]:
# Se o site tivesse o mesmo mix de plano das lojas (padronização direta pelo mix das lojas)
mix = d26[d26.onde == "loja"].groupby("plano").ativa_inicio_mes.sum()
mix = mix / mix.sum()
tx = churn(d26, ["onde", "plano"])["churn %"].unstack()
print("mix de plano na base ativa das lojas:", (mix * 100).round(1).to_dict())
print("mix de plano na base ativa do site:", (d26[d26.onde == 'site'].groupby('plano').ativa_inicio_mes.sum().pipe(lambda s: s / s.sum()) * 100).round(1).to_dict())
print("churn mensal 2026 com o mix de plano das lojas:", (tx * mix).sum(axis=1).round(2).to_dict())
print()
print("tipo de saída por canal, saídas de 2026 (% da linha)")
e = assin[assin.saiu & (assin.data_fim >= "2026-01-01")]
print(len(e), "saídas em 2026")
print(pd.crosstab(e.canal_cobranca, e.motivo_fim, normalize="index", margins=True, margins_name="total") * 100)
print(pd.crosstab(e.canal_cobranca, e.motivo_fim, margins=True, margins_name="total"))

mix de plano na base ativa das lojas: {'anual': 57.0, 'mensal': 43.0}


mix de plano na base ativa do site: {'anual': 54.8, 'mensal': 45.2}
churn mensal 2026 com o mix de plano das lojas: {'loja': 6.33, 'site': 7.08}

tipo de saída por canal, saídas de 2026 (% da linha)
8399 saídas em 2026
motivo_fim      chargeback  falha_pagamento  reembolso  renovacao_desligada
canal_cobranca                                                             
app_store              1.3              4.1        0.3                 94.4
google_play            1.4              3.2        0.6                 94.8
web_cartao             1.5             16.9        0.2                 81.4
web_pix                2.7              5.7        0.4                 91.1
total                  1.5              6.8        0.4                 91.2
motivo_fim      chargeback  falha_pagamento  reembolso  renovacao_desligada  total
canal_cobranca                                                                    
app_store               35              109          8                 2539   2691


In [5]:
# Sobrevivência: assinaturas iniciadas de 07/2024 a 09/2025, % ainda ativa no corte
c = assin[(assin.inicio >= "2024-07-01") & (assin.inicio <= "2025-09-30")]
print("assinaturas iniciadas de 07/2024 a 09/2025:", len(c))
print((c.pivot_table(index="canal_cobranca", columns="plano_atual", values="ativa_no_corte", aggfunc="mean") * 100).join(c.groupby("canal_cobranca").size().rename("assinaturas")))
print()
print("duração mediana das encerradas (dias), por canal e plano")
print(assin[assin.saiu & ~assin.inicio_antes_do_log].pivot_table(index="canal_cobranca", columns="plano_atual", values="duracao_dias", aggfunc="median"))

assinaturas iniciadas de 07/2024 a 09/2025: 14471


                anual  mensal  assinaturas
canal_cobranca                            
app_store        49.2    18.3         4910
google_play      46.8    15.7         4855
web_cartao       44.9    13.9         3190
web_pix          48.4    16.0         1516

duração mediana das encerradas (dias), por canal e plano
plano_atual     anual  mensal
canal_cobranca               
app_store       365.0   150.0
google_play     365.0   150.0
web_cartao      365.0   120.0
web_pix         365.0   150.0


## 3. Engajamento dos ativos no corte

In [6]:
e = eng.merge(assin[["user_id", "canal_cobranca", "onde", "plano_atual", "renovacao_automatica"]], on="user_id", how="left", validate="one_to_one")
print("ativos:", len(eng), "-> depois da junção:", len(e))
def niveis(d, por):
    n = pd.crosstab(d[por], d.nivel, margins=True, margins_name="total").reindex(columns=NIVEIS + ["total"])
    p_ = (pd.crosstab(d[por], d.nivel, normalize="index", margins=True, margins_name="total") * 100).reindex(columns=NIVEIS)
    return p_.join(n["total"].rename("ativos"))
print(niveis(e, "onde")); print(); print(niveis(e, "canal_cobranca"))
for pl in ["mensal", "anual"]:
    print(); print("plano", pl); print(niveis(e[e.plano_atual == pl], "canal_cobranca"))
print()
print("ativos com a renovação automática desligada (%), por canal e plano")
e["desligada"] = e.renovacao_automatica == "nao"
print(e.pivot_table(index="canal_cobranca", columns="plano_atual", values="desligada", aggfunc="mean", margins=True) * 100)
print()
print("treinos em 30 dias (média), por canal e plano")
print(e.pivot_table(index="canal_cobranca", columns="plano_atual", values="treinos_30d", aggfunc="mean", margins=True))

ativos: 14753 -> depois da junção: 14753


       1. alto  2. médio  3. baixo  4. possível churn  ativos
onde                                                         
loja      72.7      16.0       2.5                8.8   10359
site      71.8      16.2       2.5                9.5    4394
total     72.5      16.1       2.5                9.0   14753

                1. alto  2. médio  3. baixo  4. possível churn  ativos
canal_cobranca                                                        
app_store          74.5      15.3       2.3                8.0    5395
google_play        70.9      16.7       2.7                9.7    4964
web_cartao         70.0      16.9       2.8               10.3    2994
web_pix            75.4      14.9       1.9                7.9    1400
total              72.5      16.1       2.5                9.0   14753

plano mensal


                1. alto  2. médio  3. baixo  4. possível churn  ativos
canal_cobranca                                                        
app_store          86.9      12.6       0.4                0.1    2348
google_play        86.5      12.9       0.5                0.1    2036
web_cartao         85.9      13.4       0.6                0.1    1191
web_pix            85.7      13.7       0.4                0.1     700
total              86.4      13.0       0.5                0.1    6275

plano anual
                1. alto  2. médio  3. baixo  4. possível churn  ativos
canal_cobranca                                                        
app_store          64.9      17.5       3.7               14.0    3047
google_play        60.0      19.4       4.2               16.4    2928
web_cartao         59.6      19.1       4.3               17.0    1803
web_pix            65.1      16.0       3.3               15.6     700
total              62.1      18.4       4.0               15.6  